# Audio Predict Pipeline
**Full pipeline:** `.mp3` → Mel spectrogram PNGs → EfficientNet-B0 predictions

This notebook walks through each stage:
1. Audio preprocessing (load, resample, pre-emphasis, normalize, segment)
2. Frame parameter computation
3. STFT computation
4. Mel spectrogram generation
5. Saving spectrogram images
6. Loading the model & running predictions

##  Imports

In [85]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from scipy.signal import lfilter

import torch
import torch.nn as nn
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image

##  Configuration 

In [68]:
class Config:
    # Preprocessing
    TARGET_SR      = 22050
    PRE_EMPHASIS   = 0.97
    SEGMENT_LEN_S  = 20.0        # seconds per clip
    # Framing & windowing
    FRAME_LEN_MS   = 25
    HOP_LEN_MS     = 10
    # STFT
    N_FFT          = 512
    WINDOW         = "hamming"
    # Mel filterbank
    N_MELS         = 128
    F_MIN          = 0.0
    F_MAX          = None
    # Output
    COLORMAP       = "magma"
    OUTPUT_DIR     = "spectrograms"
    # Model
    MODEL_PATH     = "best_efficientnet.pth"
    IMG_SIZE       = 224
    DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
    CLASSES        = [
        "Atelectasis", "Consolidation Lung", "COVID-19", "Edema",
        "Lungs Cancer", "Normal", "Pneumonia", "Pneumothorax", "Tuberculosis"
    ]

print(f"Device: {Config.DEVICE}")
print(f"Classes: {Config.CLASSES}")

Device: cuda
Classes: ['Atelectasis', 'Consolidation Lung', 'COVID-19', 'Edema', 'Lungs Cancer', 'Normal', 'Pneumonia', 'Pneumothorax', 'Tuberculosis']


##  Step 1 : Audio Preprocessing

In [69]:
def load_and_convert_to_wav(input_path, output_path, sr=Config.TARGET_SR):
    print(f"[1/5] Loading audio: {input_path}")
    audio, orig_sr = librosa.load(input_path, sr=None, mono=True)
    print(f"      Original SR={orig_sr} Hz, duration={len(audio)/orig_sr:.2f}s")

    if orig_sr != sr:
        print(f"      Resampling {orig_sr} → {sr} Hz")
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)

    sf.write(output_path, audio, sr, subtype="PCM_16")
    print(f"      Saved WAV: {output_path}")
    return audio


def apply_pre_emphasis(audio, coeff=Config.PRE_EMPHASIS):
    print(f"[1/5] Pre-emphasis (α={coeff})")
    return lfilter([1.0, -coeff], [1.0], audio)


def normalize_audio(audio):
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    print(f"[1/5] Normalized — peak: {peak:.4f}")
    return audio


def segment_audio(audio, sr, seg_len_s=Config.SEGMENT_LEN_S):
    seg_len = int(seg_len_s * sr)
    n_segs  = max(1, int(np.ceil(len(audio) / seg_len)))
    segments = []
    for i in range(n_segs):
        start = i * seg_len
        clip  = audio[start : start + seg_len]
        if len(clip) < seg_len:
            clip = np.pad(clip, (0, seg_len - len(clip)))
        segments.append(clip)
    print(f"[1/5] Segmented → {n_segs} clip(s) × {seg_len_s}s")
    return segments

##  Step 2 : Frame Parameters

In [70]:
def get_frame_params(sr, n_fft=Config.N_FFT):
    frame_len = int(sr * Config.FRAME_LEN_MS / 1000)
    hop_len   = int(sr * Config.HOP_LEN_MS  / 1000)
    frame_len = min(2 ** int(np.ceil(np.log2(frame_len))), n_fft)
    print(f"[2/5] Frame={frame_len} samples, Hop={hop_len} samples, window={Config.WINDOW}")
    return frame_len, hop_len

##  Step 3 : STFT

In [71]:
def compute_stft(audio, n_fft, hop_len, win_len):
    print(f"[3/5] STFT — n_fft={n_fft}, hop={hop_len}, win={win_len}")
    stft     = librosa.stft(audio, n_fft=n_fft, hop_length=hop_len,
                             win_length=win_len, window=Config.WINDOW)
    power_db = librosa.power_to_db(np.abs(stft) ** 2, ref=np.max)
    print(f"      Shape (freq × time): {power_db.shape}")
    return power_db

##  Step 4 : Mel Spectrogram

In [72]:
def compute_mel_spectrogram(audio, sr, n_fft, hop_len, win_len):
    print(f"[4/5] Mel filterbank — n_mels={Config.N_MELS}")
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=n_fft,
        hop_length=hop_len, win_length=win_len,
        window=Config.WINDOW, n_mels=Config.N_MELS,
        fmin=Config.F_MIN, fmax=Config.F_MAX,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    print(f"      Shape (mels × time): {mel_db.shape}")
    return mel_db

##  Step 5 : Save Spectrogram Image

In [73]:
def save_spectrogram_image(mel_db, sr, hop_len, out_path):
    print(f"[5/5] Saving image: {out_path}")
    fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
    librosa.display.specshow(
        mel_db, sr=sr, hop_length=hop_len,
        fmin=Config.F_MIN, fmax=Config.F_MAX,
        cmap=Config.COLORMAP, ax=ax,
    )
    ax.axis("off")
    plt.subplots_adjust(0, 0, 1, 1)
    plt.savefig(out_path, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    print(f"      Saved ✓")

##  Model Load

In [74]:
def load_model(model_path=Config.MODEL_PATH, device=Config.DEVICE):
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, len(Config.CLASSES))
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"✅ Model loaded from {model_path}  (device: {device})")
    return model

##  Model Predict

In [75]:
transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])


def predict(image_path, model, device=Config.DEVICE):
    img    = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        probs   = torch.softmax(outputs, dim=1)[0]

    print(f"🖼️  Segment: {os.path.basename(image_path)} — top: {Config.CLASSES[probs.argmax().item()]} ({probs.max().item()*100:.1f}%)")
    return probs  # ← return full probability tensor, not just top class

##  Run the Full Pipeline

Set `INPUT_PATH` to your `.mp3` file and run the cell below.

In [76]:
INPUT_PATH = "Test_audio.mp3"   # ← change this to your file

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
sr       = Config.TARGET_SR
basename = os.path.splitext(os.path.basename(INPUT_PATH))[0]
wav_path = os.path.join(Config.OUTPUT_DIR, f"{basename}.wav")

# ── Audio processing ──────────────────────────────────────
audio    = load_and_convert_to_wav(INPUT_PATH, wav_path, sr=sr)
audio    = apply_pre_emphasis(audio)
audio    = normalize_audio(audio)
segments = segment_audio(audio, sr)
win_len, hop_len = get_frame_params(sr)

# ── Generate spectrograms ─────────────────────────────────
image_paths = []
for idx, segment in enumerate(segments):
    label    = f"{basename}_seg{idx+1:03d}"
    img_path = os.path.join(Config.OUTPUT_DIR, f"{label}.png")

    _      = compute_stft(segment, Config.N_FFT, hop_len, win_len)
    mel_db = compute_mel_spectrogram(segment, sr, Config.N_FFT, hop_len, win_len)
    save_spectrogram_image(mel_db, sr, hop_len, img_path)
    image_paths.append(img_path)

print(f"\n✅ {len(image_paths)} spectrogram(s) saved to '{Config.OUTPUT_DIR}/'")

[1/5] Loading audio: Test_audio.mp3


      Original SR=44100 Hz, duration=57.73s
      Resampling 44100 → 22050 Hz
      Saved WAV: spectrograms\Test_audio.wav
[1/5] Pre-emphasis (α=0.97)
[1/5] Normalized — peak: 0.5910
[1/5] Segmented → 3 clip(s) × 20.0s
[2/5] Frame=512 samples, Hop=220 samples, window=hamming
[3/5] STFT — n_fft=512, hop=220, win=512
      Shape (freq × time): (257, 2005)
[4/5] Mel filterbank — n_mels=128
      Shape (mels × time): (128, 2005)
[5/5] Saving image: spectrograms\Test_audio_seg001.png
      Saved ✓
[3/5] STFT — n_fft=512, hop=220, win=512
      Shape (freq × time): (257, 2005)
[4/5] Mel filterbank — n_mels=128
      Shape (mels × time): (128, 2005)
[5/5] Saving image: spectrograms\Test_audio_seg002.png
      Saved ✓
[3/5] STFT — n_fft=512, hop=220, win=512
      Shape (freq × time): (257, 2005)
[4/5] Mel filterbank — n_mels=128
      Shape (mels × time): (128, 2005)
[5/5] Saving image: spectrograms\Test_audio_seg003.png
      Saved ✓

✅ 3 spectrogram(s) saved to 'spectrograms/'


In [84]:
# RUNNING PREDICTIONS 

model      = load_model()
all_probs  = []

for img_path in image_paths:
    probs = predict(img_path, model)
    all_probs.append(probs)

# Average across all segments
avg_probs  = torch.stack(all_probs).mean(dim=0)
best_idx   = avg_probs.argmax().item()
best_class = Config.CLASSES[best_idx]
best_prob  = avg_probs[best_idx].item() * 100

✅ Model loaded from best_efficientnet.pth  (device: cuda)
🖼️  Segment: Test_audio_seg001.png — top: Edema (22.1%)
🖼️  Segment: Test_audio_seg002.png — top: Edema (19.2%)
🖼️  Segment: Test_audio_seg003.png — top: Edema (16.2%)


In [ ]:
# FINAL PREDICTION  (averaged over all segments)


top3 = torch.topk(avg_probs, 3)

print(f"\nTop averaged over all segments :  {best_class}  ({best_prob:.1f}%)\n")
print("Top 3:")
for prob, idx in zip(top3.values, top3.indices):
    bar = "█" * int(prob.item() * 30)
    print(f"  {Config.CLASSES[idx.item()]:25s}  {prob.item()*100:5.1f}%  {bar}")


Top averaged over all segments :  Edema  (19.2%)

Top 3:
  Edema                       19.2%  █████
  Normal                      14.3%  ████
  Consolidation Lung          10.9%  ███
